In [ ]:
import kagglehub

path = kagglehub.dataset_download(
    "vencerlanz09/shells-or-pebbles-an-image-classification-dataset"
)

print(path)

100%|██████████| 195M/195M [00:01<00:00, 129MB/s]

Extracting files...


/root/.cache/kagglehub/datasets/vencerlanz09/shells-or-pebbles-an-image-classification-dataset/versions/2


In [ ]:
from pathlib import Path

dataset_path = Path(path)

for item in dataset_path.iterdir():
    print(item)

/root/.cache/kagglehub/datasets/vencerlanz09/shells-or-pebbles-an-image-classification-dataset/versions/2/Shells
/root/.cache/kagglehub/datasets/vencerlanz09/shells-or-pebbles-an-image-classification-dataset/versions/2/Pebbles


In [ ]:
from pathlib import Path
import random
import shutil

random.seed(42)

dataset_path = Path(path)

source_dirs = {
    "Shells": dataset_path / "Shells",
    "Pebbles": dataset_path / "Pebbles"
}

train_dir = Path("data/train")
test_dir = Path("data/test")

for class_name, source_dir in source_dirs.items():

    (train_dir / class_name).mkdir(parents=True, exist_ok=True)
    (test_dir / class_name).mkdir(parents=True, exist_ok=True)

    images = list(source_dir.glob("*"))

    random.shuffle(images)

    split = int(0.8 * len(images))

    train_images = images[:split]
    test_images = images[split:]

    for img in train_images:
        shutil.copy(img, train_dir / class_name / img.name)

    for img in test_images:
        shutil.copy(img, test_dir / class_name / img.name)

print("Dataset split completed!")

Dataset split completed!


In [ ]:
import torch
from torchvision import datasets
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from torch.utils.data import DataLoader

In [ ]:
weights = EfficientNet_B0_Weights.DEFAULT

auto_transforms = weights.transforms()

In [ ]:
train_data = datasets.ImageFolder(
    root="data/train",
    transform=auto_transforms
)

test_data = datasets.ImageFolder(
    root="data/test",
    transform=auto_transforms
)

class_names = train_data.classes

print(class_names)

['Pebbles', 'Shells']


In [ ]:
train_loader = DataLoader(
    train_data,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_data,
    batch_size=32,
    shuffle=False
)

In [ ]:
model = efficientnet_b0(weights=weights).to(device)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 125MB/s]


In [ ]:
for param in model.features.parameters():
    param.requires_grad = False

In [ ]:
import torch.nn as nn

model.classifier = nn.Sequential(
    nn.Dropout(0.2),
    nn.Linear(
        in_features=1280,
        out_features=len(class_names)
    )
).to(device)

In [ ]:
import torch.nn as nn

loss_fn = nn.CrossEntropyLoss().to(device)

model.eval()

test_loss = 0
test_acc = 0

with torch.inference_mode():
    for X, y in test_loader:
        X, y = X.to(device), y.to(device)

        test_pred = model(X)

        loss = loss_fn(test_pred, y)
        test_loss += loss.item()

        pred_labels = torch.argmax(test_pred, dim=1)

        test_acc += (pred_labels == y).sum().item() / len(pred_labels)

test_loss /= len(test_loader)
test_acc /= len(test_loader)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc*100:.2f}%")

Test Loss: 0.6767
Test Accuracy: 55.92%


In [ ]:
from PIL import Image

image = Image.open("sample.jpg")

image = auto_transforms(image).unsqueeze(0).to(device)

model.eval()

with torch.inference_mode():
    pred = model(image)
    pred_class = torch.argmax(pred, dim=1)

print("Predicted class:", class_names[pred_class])

Predicted class: Pebbles


In [16]:
import torch.optim as optim

optimizer = optim.Adam(model.parameters(), lr=0.001)

In [17]:
def train_step(model, dataloader, loss_fn, optimizer, device):
    model.train()
    train_loss, train_acc = 0, 0
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # 1. Forward pass
        y_pred = model(X)

        # 2. Calculate loss
        loss = loss_fn(y_pred, y)
        train_loss += loss.item()

        # 3. Optimizer zero grad
        optimizer.zero_grad()

        # 4. Loss backward
        loss.backward()

        # 5. Optimizer step
        optimizer.step()

        # Calculate accuracy
        y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
        train_acc += (y_pred_class == y).sum().item()/len(y_pred)

    # Adjust metrics to get average loss and accuracy per batch
    train_loss = train_loss / len(dataloader)
    train_acc = train_acc / len(dataloader)
    return train_loss, train_acc

In [18]:
def test_step(model, dataloader, loss_fn, device):
    model.eval()
    test_loss, test_acc = 0, 0
    with torch.inference_mode():
        for batch, (X, y) in enumerate(dataloader):
            X, y = X.to(device), y.to(device)

            # 1. Forward pass
            test_pred_logits = model(X)

            # 2. Calculate loss
            loss = loss_fn(test_pred_logits, y)
            test_loss += loss.item()

            # Calculate accuracy
            test_pred_labels = test_pred_logits.argmax(dim=1)
            test_acc += ((test_pred_labels == y).sum().item()/len(test_pred_labels))

    # Adjust metrics to get average loss and accuracy per batch
    test_loss = test_loss / len(dataloader)
    test_acc = test_acc / len(dataloader)
    return test_loss, test_acc

In [19]:
from tqdm.auto import tqdm

epochs = 5

for epoch in tqdm(range(epochs)):
    train_loss, train_acc = train_step(model, train_loader, loss_fn, optimizer, device)
    test_loss, test_acc = test_step(model, test_loader, loss_fn, device)

    print(
        f"Epoch: {epoch+1} | "
        f"train_loss: {train_loss:.4f} | "
        f"train_acc: {train_acc:.4f} | "
        f"test_loss: {test_loss:.4f} | "
        f"test_acc: {test_acc:.4f}"
    )

print("\nTraining complete!")

  0%|          | 0/5 [00:00<?, ?it/s]

Epoch: 1 | train_loss: 0.4481 | train_acc: 0.7995 | test_loss: 0.3538 | test_acc: 0.8603
Epoch: 2 | train_loss: 0.3677 | train_acc: 0.8429 | test_loss: 0.3427 | test_acc: 0.8661
Epoch: 3 | train_loss: 0.3403 | train_acc: 0.8539 | test_loss: 0.3447 | test_acc: 0.8566
Epoch: 4 | train_loss: 0.3286 | train_acc: 0.8686 | test_loss: 0.3277 | test_acc: 0.8594
Epoch: 5 | train_loss: 0.3180 | train_acc: 0.8660 | test_loss: 0.3300 | test_acc: 0.8661

Training complete!
